# Quantitativer Vergleich: Crop vs. Red Circle auf VOC (N=1000)

Ziel: Messung der Konzeptqualität beider Methoden auf Image-Ebene.

**Pipeline:**
1. VOC-Subset (1000 Bilder) mit GT-Annotationen aus XML
2. Für jedes Bild: Full Image + alle GT-Objekte mit Crop bzw. Circle enkodieren
3. Aggregation: `full_emb + sum(object_embs)`
4. Metrik: **Hit@K** — erscheint irgendein GT-Label in den Top-K SpLiCE-Konzepten?s

In [ ]:
import sys
sys.path.insert(0, "/workspaces/Object-Centric-Concept-Bottlenecks")

import os
import torch
import open_clip
import splice
from splice.model import SPLICE
from PIL import Image, ImageDraw
import torchvision.transforms.functional as TF
import xml.etree.ElementTree as ET
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Modelle laden

SpLiCE mit LAION-Backbone (ViT-B-32) — gleiche Konfiguration wie im qualitativen Notebook.

In [ ]:
concepts   = torch.load("/root/.cache/splice/embeddings/open_clip:ViT-B-32_laion_10000_embeddings.pt", map_location=device)
image_mean = torch.load("/root/.cache/splice/means/open_clip_ViT-B-32_image.pt", map_location=device)

clip_backbone, _, splice_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_backbone = clip_backbone.to(device)
clip_backbone.eval()

splice_model = SPLICE(
    image_mean=image_mean, dictionary=concepts, clip=clip_backbone,
    device=device, l1_penalty=0.25, return_weights=True,
)
splice_model.eval()

vocab = splice.get_vocabulary("laion", vocabulary_size=10000)
print(f"SpLiCE geladen | Vocab: {len(vocab)} Konzepte | Device: {device}")

## Hilfsfunktionen

- `draw_circle_on_image`: roter Ellipsen-Kreis um Bounding Box (Shtedritski et al., ICCV 2023)
- `crop_object`: Objekt ausschneiden — kein globaler Kontext
- `get_splice_embedding`: SpLiCE-Embedding für ein PIL-Bild
- `label_hit`: prüft ob GT-Label (fuzzy) in Top-K Konzepten auftaucht
- `label_rank`: gibt den Rang des besten GT-Labels zurück

In [ ]:
# Manually curated synonyms for each VOC class, filtered to words present in the SpLiCE vocabulary.

VOC_SYNONYMS = {
    "aeroplane":   ["airplane", "aircraft", "plane", "planes", "jet", "jets",
                    "aviation", "airline", "airlines", "airplanes"],
    "bicycle":     ["bicycle", "bicycles", "bike", "bikes", "cycling", "cyclist", "cyclists"],
    "bird":        ["bird", "birds", "sparrow", "eagle", "robin", "parrot", "owl"],
    "boat":        ["boat", "boats", "ship", "ships", "vessel", "yacht", "sailing", "canoe", "kayak", "ferry"],
    "bottle":      ["bottle", "bottles", "flask", "jar", "jug"],
    "bus":         ["bus", "buses", "coach", "coaches", "shuttle", "transit"],
    "car":         ["car", "cars", "automobile", "vehicle", "vehicles", "sedan", "auto", "autos", "coupe", "hatchback", "suv"],
    "cat":         ["cat", "cats", "kitten", "kittens", "kitty"],
    "chair":       ["chair", "chairs", "seat", "seats", "stool", "seating"],
    "cow":         ["cow", "cows", "cattle", "bull", "bulls", "ox", "calf", "dairy"],
    "diningtable": ["dining", "table", "tables", "desk"],
    "dog":         ["dog", "dogs", "puppy", "puppies", "canine", "hound", "pup"],
    "horse":       ["horse", "horses", "pony", "stallion", "mare"],
    "motorbike":   ["motorbike", "motorcycle", "motorcycles"],
    "person":      ["person", "persons", "people", "man", "woman", "men", "women",
                    "human", "humans", "boy", "girl", "child", "children", "individual"],
    "pottedplant": ["plant", "plants", "potted", "flower", "flowers", "succulent"],
    "sheep":       ["sheep", "lamb", "ram", "flock", "wool"],
    "sofa":        ["sofa", "couch", "sectional"],
    "train":       ["train", "trains", "locomotive", "rail", "railway", "railroad", "metro", "subway"],
    "tvmonitor":   ["tv", "television", "monitor", "screen", "display"],
}


for cls, syns in VOC_SYNONYMS.items():
    found = [s for s in syns if s in vocab]
    print(f"{cls:<15}: {found}")


In [ ]:
def draw_circle_on_image(image_tensor, box, color=(255, 0, 0), width=6):
    image_pil = TF.to_pil_image(image_tensor.cpu())
    draw = ImageDraw.Draw(image_pil)
    x1, y1, x2, y2 = box.long()
    draw.ellipse([x1.item(), y1.item(), x2.item(), y2.item()], outline=color, width=width)
    return image_pil

def crop_object(image_tensor, box):
    x1, y1, x2, y2 = box.long()
    return TF.to_pil_image(image_tensor[:, y1:y2, x1:x2].cpu())

def get_splice_embedding(image_pil, model, preprocess):
    img_t = preprocess(image_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        emb, _ = model.encode_image(img_t)
    return emb.squeeze(0)

def expand_labels(gt_labels):
    expanded = set()
    for lbl in gt_labels:
        expanded.update(VOC_SYNONYMS.get(lbl, [lbl]))
    return expanded

def label_hit(emb, vocab, gt_labels, top_k):
    expanded = expand_labels(gt_labels)
    top_words = [vocab[i] for i in emb.argsort(descending=True)[:top_k]]
    return any(lbl in w or w in lbl for w in top_words for lbl in expanded)

def label_rank(emb, vocab, gt_labels, search_k=100):
    expanded = expand_labels(gt_labels)
    for rank, i in enumerate(emb.argsort(descending=True)[:search_k]):
        w = vocab[i]
        if any(lbl in w or w in lbl for lbl in expanded):
            return rank + 1
    return None


print("Hilfsfunktionen definiert")


## VOC-Subset laden

1000 Bilder aus dem VOC2012 Validation-Set.  
GT-Labels direkt aus XML-Annotationen 

In [ ]:
IMG_DIR = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/JPEGImages"
ANN_DIR = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/Annotations"
IMGSET  = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/ImageSets/Main/val.txt"

with open(IMGSET) as f:
    subset_ids = [l.strip() for l in f if l.strip()][:1000]

print(f"Subset: {len(subset_ids)} Bilder")

## Evaluation Loop

Für jedes Bild: GT-Boxen aus XML laden, Crop- und Circle-Embeddings berechnen,  
mit Sum über alle Objekte + Full Image aggregieren.  

In [ ]:
results = []

for img_id in tqdm(subset_ids, desc="Evaluating"):
    ann_path = os.path.join(ANN_DIR, f"{img_id}.xml")
    img_path = os.path.join(IMG_DIR,  f"{img_id}.jpg")
    if not os.path.exists(ann_path) or not os.path.exists(img_path):
        continue

    tree_i = ET.parse(ann_path)
    boxes_i, labels_i = [], []
    for obj in tree_i.getroot().findall("object"):
        bb = obj.find("bndbox")
        boxes_i.append(torch.tensor([
            int(bb.find("xmin").text), int(bb.find("ymin").text),
            int(bb.find("xmax").text), int(bb.find("ymax").text),
        ]))
        labels_i.append(obj.find("name").text)
    if not boxes_i:
        continue

    pil_i    = Image.open(img_path).convert("RGB")
    tensor_i = TF.to_tensor(pil_i)
    full_e   = get_splice_embedding(pil_i, splice_model, splice_preprocess)

    crop_embs_i   = torch.stack([get_splice_embedding(crop_object(tensor_i, b),
                                  splice_model, splice_preprocess) for b in boxes_i])
    circle_embs_i = torch.stack([get_splice_embedding(draw_circle_on_image(tensor_i, b),
                                  splice_model, splice_preprocess) for b in boxes_i])

    crop_stack_i   = torch.cat([full_e.unsqueeze(0), crop_embs_i],   dim=0)
    circle_stack_i = torch.cat([full_e.unsqueeze(0), circle_embs_i], dim=0)

    results.append({
        "gt":           list(set(labels_i)),
        "n_objects":    len(boxes_i),
        "crop_sum":     full_e + crop_embs_i.sum(dim=0),
        "circle_sum":   full_e + circle_embs_i.sum(dim=0),
        "crop_count":   (crop_stack_i   > 0).float().sum(dim=0),
        "circle_count": (circle_stack_i > 0).float().sum(dim=0),
    })


print(f"Evaluated {len(results)} Bilder")

## Ergebnisse

- **Hit@K**: Anteil der Bilder, bei denen mind. ein GT-Label in Top-K Konzepten auftaucht
- **MeanRank**: Mittlerer Rang des besten GT-Labels (nur über gefundene Bilder)
- **Found/N**: Wie viele Bilder ein GT-Match in Top-100 hatten

bidirektionaler Substring-Check: `"aeroplane" in "aeroplane"` oder `"plane" in "aeroplane"`.

In [ ]:
n = len(results)
print(f"{'Methode':<20} {'Hit@10':>8} {'Hit@20':>8} {'MeanRank':>10} {'Found':>8}")
print("-" * 52)

for key, label in [("crop_sum", "Crop + Sum"), ("circle_sum", "Circle + Sum"), ("crop_count",   "Crop   + Count"),
    ("circle_count", "Circle + Count")]:
    hits10 = sum(label_hit(r[key], vocab, r["gt"], 10) for r in results)
    hits20 = sum(label_hit(r[key], vocab, r["gt"], 20) for r in results)
    ranks  = [label_rank(r[key], vocab, r["gt"]) for r in results]
    found  = [rk for rk in ranks if rk is not None]
    mean_r = sum(found) / len(found) if found else float("nan")
    print(f"{label:<20} {hits10/n:>8.3f} {hits20/n:>8.3f} {mean_r:>10.1f} {len(found):>4}/{n}")

## COCO-Subset (N=500)

Gleiche Pipeline, andere Annotationsstruktur: JSON statt XML,  
Bounding Box Format `[x, y, w, h]` → `[x1, y1, x2, y2]`.

In [ ]:
COCO_SYNONYMS = {
    "person":        ["person", "persons", "people", "man", "woman", "men", "women",
                      "human", "humans", "boy", "girl", "child", "children", "individual"],
    "bicycle":       ["bicycle", "bicycles", "bike", "bikes", "cycling", "cyclist", "cyclists"],
    "car":           ["car", "cars", "automobile", "vehicle", "vehicles", "sedan",
                      "auto", "autos", "coupe", "hatchback", "suv"],
    "motorcycle":    ["motorcycle", "motorcycles", "motorbike"],
    "airplane":      ["airplane", "airplanes", "aircraft", "plane", "planes", "jet", "jets",
                      "aviation", "airline", "airlines"],
    "bus":           ["bus", "buses", "coach", "coaches", "shuttle", "transit"],
    "train":         ["train", "trains", "locomotive", "rail", "railway", "railroad", "metro", "subway"],
    "truck":         ["truck", "trucks", "lorry", "van", "pickup", "freight"],
    "boat":          ["boat", "boats", "ship", "ships", "vessel", "yacht", "sailing",
                      "canoe", "kayak", "ferry"],
    "traffic light": ["traffic", "signal", "intersection"],
    "fire hydrant":  [],
    "stop sign":     ["sign", "signs", "signage"],
    "parking meter": ["parking", "meter"],
    "bench":         ["bench", "benches", "seat", "seats", "seating"],
    "bird":          ["bird", "birds", "sparrow", "eagle", "robin", "parrot", "owl"],
    "cat":           ["cat", "cats", "kitten", "kittens", "kitty"],
    "dog":           ["dog", "dogs", "puppy", "puppies", "canine", "hound", "pup"],
    "horse":         ["horse", "horses", "pony", "stallion", "mare"],
    "sheep":         ["sheep", "lamb", "ram", "flock", "wool"],
    "cow":           ["cow", "cows", "cattle", "bull", "bulls", "ox", "calf", "dairy"],
    "elephant":      ["elephant", "elephants"],
    "bear":          ["bear", "bears", "grizzly", "polar", "panda"],
    "zebra":         ["zebra"],
    "giraffe":       ["giraffe"],
    "backpack":      ["backpack", "backpacks", "bag", "bags"],
    "umbrella":      ["umbrella", "umbrellas"],
    "handbag":       ["handbag", "handbags", "purse", "bag", "bags"],
    "tie":           ["tie", "ties"],
    "suitcase":      ["suitcase", "luggage", "baggage"],
    "frisbee":       ["disc", "disk"],
    "skis":          ["skis", "ski", "skiing", "skier"],
    "snowboard":     ["snowboard"],
    "sports ball":   ["ball", "balls", "soccer", "football", "basketball", "baseball"],
    "kite":          ["kite"],
    "baseball bat":  ["bat", "bats", "baseball"],
    "baseball glove":["glove", "gloves", "baseball"],
    "skateboard":    ["skateboard", "skater"],
    "surfboard":     ["surfing", "surfer"],
    "tennis racket": ["tennis", "racket"],
    "bottle":        ["bottle", "bottles", "flask", "jar", "jug"],
    "wine glass":    ["wine", "glass", "glasses"],
    "cup":           ["cup", "cups", "mug", "mugs"],
    "fork":          ["fork", "cutlery"],
    "knife":         ["knife", "knives", "blade", "cutlery"],
    "spoon":         ["spoon", "spoons", "cutlery"],
    "bowl":          ["bowl", "bowls", "dish", "dishes"],
    "banana":        ["banana", "bananas"],
    "apple":         ["apple", "apples"],
    "sandwich":      ["sandwich", "sandwiches"],
    "orange":        ["orange", "oranges"],
    "broccoli":      ["broccoli"],
    "carrot":        ["carrot", "carrots"],
    "hot dog":       ["sausage"],
    "pizza":         ["pizza"],
    "donut":         ["donut", "donuts"],
    "cake":          ["cake", "cakes", "pastry"],
    "chair":         ["chair", "chairs", "seat", "seats", "stool", "seating"],
    "couch":         ["couch", "sofa", "sectional"],
    "potted plant":  ["plant", "plants", "potted", "flower", "flowers", "succulent"],
    "bed":           ["bed", "beds", "bedding", "bedroom", "mattress"],
    "dining table":  ["dining", "table", "tables", "desk"],
    "toilet":        ["toilet", "bathroom", "wc"],
    "tv":            ["tv", "television", "monitor", "screen", "display"],
    "laptop":        ["laptop", "laptops", "notebook"],
    "mouse":         ["mouse", "mice"],
    "remote":        ["remote", "controller"],
    "keyboard":      ["keyboard", "typing"],
    "cell phone":    ["phone", "phones", "smartphone", "mobile", "iphone"],
    "microwave":     ["microwave"],
    "oven":          ["oven", "stove"],
    "toaster":       [],
    "sink":          ["sink", "sinks", "basin"],
    "refrigerator":  ["refrigerator", "fridge", "freezer"],
    "book":          ["book", "books", "bookshelf", "reading"],
    "clock":         ["clock", "clocks", "watch", "timer"],
    "vase":          ["vase", "vases", "pot", "pottery"],
    "scissors":      ["scissors"],
    "teddy bear":    ["teddy", "stuffed", "plush", "toy"],
    "hair drier":    ["dryer"],
    "toothbrush":    ["toothbrush"],
}
for cls, syns in COCO_SYNONYMS.items():
    found = [s for s in syns if s in vocab]
    print(f"{cls:<20}: {found}")


In [ ]:
import json

IMG_DIR_COCO  = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/images/val2017"
ANN_PATH_COCO = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/annotations/instances_val2017.json"

with open(ANN_PATH_COCO) as f:
    coco_data = json.load(f)

cat_id_to_name  = {c["id"]: c["name"] for c in coco_data["categories"]}
img_id_to_fname = {img["id"]: img["file_name"] for img in coco_data["images"]}

img_id_to_anns = {}
for ann in coco_data["annotations"]:
    iid = ann["image_id"]
    x, y, w, h = ann["bbox"]
    img_id_to_anns.setdefault(iid, []).append((
        torch.tensor([x, y, x + w, y + h]),
        cat_id_to_name[ann["category_id"]]
    ))

subset_img_ids = [
    img["id"] for img in coco_data["images"]
    if img["id"] in img_id_to_anns
][:500]
print(f"Subset: {len(subset_img_ids)} Bilder")

results_coco = []

for img_id in tqdm(subset_img_ids, desc="COCO Evaluating"):
    img_path = os.path.join(IMG_DIR_COCO, img_id_to_fname[img_id])
    if not os.path.exists(img_path):
        continue

    anns     = img_id_to_anns[img_id]
    boxes_i  = [a[0] for a in anns]
    labels_i = [a[1] for a in anns]

    pil_i    = Image.open(img_path).convert("RGB")
    tensor_i = TF.to_tensor(pil_i)
    full_e   = get_splice_embedding(pil_i, splice_model, splice_preprocess)

    crop_embs_i   = torch.stack([get_splice_embedding(crop_object(tensor_i, b),
                                  splice_model, splice_preprocess) for b in boxes_i])
    circle_embs_i = torch.stack([get_splice_embedding(draw_circle_on_image(tensor_i, b),
                                  splice_model, splice_preprocess) for b in boxes_i])

    crop_stack_i   = torch.cat([full_e.unsqueeze(0), crop_embs_i],   dim=0)
    circle_stack_i = torch.cat([full_e.unsqueeze(0), circle_embs_i], dim=0)

    results_coco.append({
        "gt":           list(set(labels_i)),
        "n_objects":    len(boxes_i),
        "crop_sum":     full_e + crop_embs_i.sum(dim=0),
        "circle_sum":   full_e + circle_embs_i.sum(dim=0),
        "crop_count":   (crop_stack_i   > 0).float().sum(dim=0),
        "circle_count": (circle_stack_i > 0).float().sum(dim=0),
    })

print(f"Evaluated {len(results_coco)} Bilder")

### Ergebnisse COCO

In [ ]:
n = len(results_coco)
print(f"{'Methode':<22} {'Hit@10':>8} {'Hit@20':>8} {'MeanRank':>10} {'Found':>8}")
print("-" * 58)

for key, label in [
    ("crop_sum",     "Crop   + Sum"),
    ("circle_sum",   "Circle + Sum"),
    ("crop_count",   "Crop   + Count"),
    ("circle_count", "Circle + Count"),
]:
    hits10 = sum(label_hit(r[key], vocab, r["gt"], 10) for r in results_coco)
    hits20 = sum(label_hit(r[key], vocab, r["gt"], 20) for r in results_coco)
    ranks  = [label_rank(r[key], vocab, r["gt"]) for r in results_coco]
    found  = [rk for rk in ranks if rk is not None]
    mean_r = sum(found) / len(found) if found else float("nan")
    print(f"{label:<22} {hits10/n:>8.3f} {hits20/n:>8.3f} {mean_r:>10.1f} {len(found):>4}/{n}")